# Attrition Trend Analysis

## Objective
Identify patterns in who leaves, when, and from where to pinpoint attrition hot spots.

## Key Questions
1. What is the overall attrition rate and trend?
2. Which departments/teams have highest attrition?
3. What tenure groups are most at risk?
4. Are there differences in voluntary vs. involuntary attrition?
5. What is the regrettable attrition rate?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
employees = pd.read_csv('../data/employees.csv')

print(f"Loaded {len(employees)} employee records")
print(f"  Active: {len(employees[employees['status']=='Active'])}")
print(f"  Terminated: {len(employees[employees['status']=='Terminated'])}")

## 1. Overall Attrition Metrics

In [ ]:
# Calculate key attrition metrics
active = employees[employees['status'] == 'Active']
termed = employees[employees['status'] == 'Terminated']

total_headcount = len(employees)
overall_attrition_rate = (len(termed) / total_headcount) * 100

# Voluntary vs. Involuntary
voluntary = termed[termed['termination_reason'].str.contains('Voluntary', na=False)]
involuntary = termed[termed['termination_reason'].str.contains('Involuntary', na=False)]
retirement = termed[termed['termination_reason'] == 'Retirement']

voluntary_rate = (len(voluntary) / total_headcount) * 100
involuntary_rate = (len(involuntary) / total_headcount) * 100

# Regrettable attrition
regrettable = termed[termed['regrettable_attrition'] == 1]
regrettable_rate = (len(regrettable) / total_headcount) * 100

print("ATTRITION SUMMARY")
print("="*60)
print(f"Overall Attrition Rate: {overall_attrition_rate:.1f}%")
print(f"  Voluntary: {voluntary_rate:.1f}% ({len(voluntary)} employees)")
print(f"  Involuntary: {involuntary_rate:.1f}% ({len(involuntary)} employees)")
print(f"  Retirement: {len(retirement)} employees")
print(f"\nRegrettable Attrition Rate: {regrettable_rate:.1f}% ({len(regrettable)} employees)")
print(f"  {len(regrettable)/len(termed)*100:.1f}% of all departures are regrettable")
print("="*60)

# Industry benchmark comparison
print("\nINDUSTRY BENCHMARKS (Technology Sector)")
print("  Target Overall Attrition: <12% annually")
print("  Target Voluntary Attrition: <10% annually")
print("  Target Regrettable Attrition: <5% annually")

if overall_attrition_rate > 12:
    print(f"\n⚠️  Current attrition ({overall_attrition_rate:.1f}%) is ABOVE industry target")
else:
    print(f"\n✅ Current attrition ({overall_attrition_rate:.1f}%) is within industry target")

if regrettable_rate > 5:
    print(f"⚠️  Regrettable attrition ({regrettable_rate:.1f}%) is ABOVE target - focus area!")
else:
    print(f"✅ Regrettable attrition ({regrettable_rate:.1f}%) is within target")

In [ ]:
# Visualize attrition breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: Status breakdown
status_counts = employees['status'].value_counts()
colors = ['#2ecc71', '#e74c3c']
ax1.pie(status_counts, labels=status_counts.index, autopct='%1.1f%%', 
        startangle=90, colors=colors)
ax1.set_title('Employee Status Distribution', fontsize=14, fontweight='bold')

# Bar chart: Termination type breakdown
term_types = pd.DataFrame({
    'Type': ['Voluntary', 'Involuntary', 'Retirement'],
    'Count': [len(voluntary), len(involuntary), len(retirement)]
})
bars = ax2.bar(term_types['Type'], term_types['Count'], color=['#3498db', '#e67e22', '#95a5a6'])
ax2.set_title('Termination Type Breakdown', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Employees')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Attrition by Department

In [ ]:
# Calculate attrition rate by department
dept_analysis = employees.groupby('department').agg({
    'employee_id': 'count',
    'status': lambda x: (x == 'Terminated').sum()
}).rename(columns={'employee_id': 'total_employees', 'status': 'terminations'})

dept_analysis['attrition_rate'] = (dept_analysis['terminations'] / dept_analysis['total_employees'] * 100).round(1)
dept_analysis = dept_analysis.sort_values('attrition_rate', ascending=False)

print("ATTRITION RATE BY DEPARTMENT")
print("="*60)
print(dept_analysis[['total_employees', 'terminations', 'attrition_rate']])
print("="*60)

# Identify high-attrition departments
avg_attrition = dept_analysis['attrition_rate'].mean()
high_attrition_depts = dept_analysis[dept_analysis['attrition_rate'] > avg_attrition + 5]

if len(high_attrition_depts) > 0:
    print(f"\n⚠️  HIGH ATTRITION DEPARTMENTS (>5% above average):")
    for dept in high_attrition_depts.index:
        rate = high_attrition_depts.loc[dept, 'attrition_rate']
        count = high_attrition_depts.loc[dept, 'terminations']
        print(f"  • {dept}: {rate}% ({int(count)} departures)")

In [ ]:
# Visualize department attrition rates
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(dept_analysis.index, dept_analysis['attrition_rate'], 
               color=['#e74c3c' if x > avg_attrition + 5 else '#3498db' 
                      for x in dept_analysis['attrition_rate']])

# Add average line
ax.axvline(avg_attrition, color='gray', linestyle='--', linewidth=2, 
           label=f'Average: {avg_attrition:.1f}%')

ax.set_xlabel('Attrition Rate (%)', fontsize=12)
ax.set_title('Attrition Rate by Department', fontsize=14, fontweight='bold')
ax.legend()

# Add value labels
for i, (idx, row) in enumerate(dept_analysis.iterrows()):
    ax.text(row['attrition_rate'] + 0.5, i, f"{row['attrition_rate']}%", 
            va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Attrition by Tenure

In [ ]:
# Create tenure buckets
def tenure_bucket(years):
    if years < 0.25:
        return '0-3 months'
    elif years < 1:
        return '3-12 months'
    elif years < 2:
        return '1-2 years'
    elif years < 3:
        return '2-3 years'
    elif years < 5:
        return '3-5 years'
    else:
        return '5+ years'

employees['tenure_bucket'] = employees['tenure_years'].apply(tenure_bucket)

# Calculate attrition by tenure
tenure_order = ['0-3 months', '3-12 months', '1-2 years', '2-3 years', '3-5 years', '5+ years']
tenure_analysis = employees.groupby('tenure_bucket').agg({
    'employee_id': 'count',
    'status': lambda x: (x == 'Terminated').sum()
}).rename(columns={'employee_id': 'total_employees', 'status': 'terminations'})

tenure_analysis['attrition_rate'] = (tenure_analysis['terminations'] / tenure_analysis['total_employees'] * 100).round(1)
tenure_analysis = tenure_analysis.reindex(tenure_order)

print("ATTRITION RATE BY TENURE")
print("="*60)
print(tenure_analysis[['total_employees', 'terminations', 'attrition_rate']])
print("="*60)

# Identify high-risk tenure periods
print("\nKEY INSIGHTS:")
early_attrition = tenure_analysis.loc['0-3 months', 'attrition_rate']
if early_attrition > 10:
    print(f"⚠️  HIGH EARLY ATTRITION: {early_attrition}% leave in first 90 days")
    print("   → Review hiring and onboarding processes")

midterm_attrition = tenure_analysis.loc[['1-2 years', '2-3 years'], 'attrition_rate'].mean()
if midterm_attrition > 25:
    print(f"⚠️  HIGH MID-TENURE ATTRITION: {midterm_attrition:.1f}% leave at 1-3 years")
    print("   → Focus on career development and growth opportunities")

In [ ]:
# Visualize tenure attrition
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart: Attrition rate by tenure
ax1.bar(range(len(tenure_analysis)), tenure_analysis['attrition_rate'], 
        color='#3498db', alpha=0.7)
ax1.set_xticks(range(len(tenure_analysis)))
ax1.set_xticklabels(tenure_analysis.index, rotation=45, ha='right')
ax1.set_ylabel('Attrition Rate (%)')
ax1.set_title('Attrition Rate by Tenure', fontsize=14, fontweight='bold')

# Add value labels
for i, val in enumerate(tenure_analysis['attrition_rate']):
    ax1.text(i, val + 1, f'{val}%', ha='center', fontweight='bold')

# Stacked bar: Employee distribution
termed_by_tenure = employees[employees['status']=='Terminated'].groupby('tenure_bucket').size().reindex(tenure_order, fill_value=0)
active_by_tenure = employees[employees['status']=='Active'].groupby('tenure_bucket').size().reindex(tenure_order, fill_value=0)

x = range(len(tenure_order))
ax2.bar(x, active_by_tenure, label='Active', color='#2ecc71', alpha=0.8)
ax2.bar(x, termed_by_tenure, bottom=active_by_tenure, label='Terminated', color='#e74c3c', alpha=0.8)

ax2.set_xticks(x)
ax2.set_xticklabels(tenure_order, rotation=45, ha='right')
ax2.set_ylabel('Number of Employees')
ax2.set_title('Employee Distribution by Tenure', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Attrition by Job Level

In [ ]:
# Calculate attrition by job level
level_analysis = employees.groupby('job_level').agg({
    'employee_id': 'count',
    'status': lambda x: (x == 'Terminated').sum()
}).rename(columns={'employee_id': 'total_employees', 'status': 'terminations'})

level_analysis['attrition_rate'] = (level_analysis['terminations'] / level_analysis['total_employees'] * 100).round(1)
level_analysis = level_analysis.sort_values('attrition_rate', ascending=False)

print("ATTRITION RATE BY JOB LEVEL")
print("="*60)
print(level_analysis[['total_employees', 'terminations', 'attrition_rate']])
print("="*60)

In [ ]:
# Visualize job level attrition
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(range(len(level_analysis)), level_analysis['attrition_rate'], 
              color='#9b59b6', alpha=0.7)
ax.set_xticks(range(len(level_analysis)))
ax.set_xticklabels(level_analysis.index, rotation=45, ha='right')
ax.set_ylabel('Attrition Rate (%)')
ax.set_title('Attrition Rate by Job Level', fontsize=14, fontweight='bold')

# Add value labels
for i, val in enumerate(level_analysis['attrition_rate']):
    ax.text(i, val + 1, f'{val}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Regrettable vs. Non-Regrettable Attrition

In [ ]:
# Analyze regrettable attrition
regrettable_df = termed[termed['regrettable_attrition'] == 1]
non_regrettable_df = termed[termed['regrettable_attrition'] == 0]

print("REGRETTABLE ATTRITION ANALYSIS")
print("="*60)
print(f"Total Departures: {len(termed)}")
print(f"  Regrettable: {len(regrettable_df)} ({len(regrettable_df)/len(termed)*100:.1f}%)")
print(f"  Non-Regrettable: {len(non_regrettable_df)} ({len(non_regrettable_df)/len(termed)*100:.1f}%)")
print("\nDefinition of Regrettable:")
print("  • High performer (rating ≥3.8) AND")
print("  • Voluntary departure")
print("="*60)

# Compare characteristics
print("\nCHARACTERISTICS COMPARISON")
print("="*60)
comparison = pd.DataFrame({
    'Regrettable': [
        regrettable_df['performance_rating'].mean(),
        regrettable_df['market_percentile'].mean(),
        regrettable_df['months_since_promotion'].mean(),
        regrettable_df['engagement_score'].mean(),
        regrettable_df['tenure_years'].mean()
    ],
    'Non-Regrettable': [
        non_regrettable_df['performance_rating'].mean(),
        non_regrettable_df['market_percentile'].mean(),
        non_regrettable_df['months_since_promotion'].mean(),
        non_regrettable_df['engagement_score'].mean(),
        non_regrettable_df['tenure_years'].mean()
    ]
}, index=['Avg Performance Rating', 'Avg Market Percentile', 'Avg Months Since Promotion', 
          'Avg Engagement Score', 'Avg Tenure (years)'])

print(comparison.round(2))
print("="*60)

# Regrettable attrition by department
regret_by_dept = regrettable_df.groupby('department').size().sort_values(ascending=False)
print("\nREGRETTABLE DEPARTURES BY DEPARTMENT")
print(regret_by_dept)

In [ ]:
# Visualize regrettable attrition
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart: Regrettable vs. Non-Regrettable
sizes = [len(regrettable_df), len(non_regrettable_df)]
labels = ['Regrettable', 'Non-Regrettable']
colors = ['#e74c3c', '#95a5a6']
explode = (0.1, 0)

ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90)
ax1.set_title('Attrition Breakdown: Regrettable vs. Non-Regrettable', 
              fontsize=14, fontweight='bold')

# Bar chart: Regrettable departures by department
top_regret_depts = regret_by_dept.head(8)
ax2.barh(range(len(top_regret_depts)), top_regret_depts.values, color='#e74c3c', alpha=0.7)
ax2.set_yticks(range(len(top_regret_depts)))
ax2.set_yticklabels(top_regret_depts.index)
ax2.set_xlabel('Number of Regrettable Departures')
ax2.set_title('Top Departments: Regrettable Attrition', fontsize=14, fontweight='bold')

# Add value labels
for i, val in enumerate(top_regret_depts.values):
    ax2.text(val + 0.3, i, str(val), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Termination Reason Analysis

In [ ]:
# Analyze termination reasons (voluntary only)
voluntary_reasons = voluntary['termination_reason'].value_counts()

print("TOP VOLUNTARY TERMINATION REASONS")
print("="*60)
for reason, count in voluntary_reasons.items():
    pct = (count / len(voluntary)) * 100
    print(f"{reason}: {count} ({pct:.1f}%)")
print("="*60)

In [ ]:
# Visualize termination reasons
fig, ax = plt.subplots(figsize=(12, 8))

# Clean up labels for display
labels = [reason.replace('Voluntary - ', '') for reason in voluntary_reasons.index]
values = voluntary_reasons.values

bars = ax.barh(range(len(voluntary_reasons)), values, color='#3498db', alpha=0.7)
ax.set_yticks(range(len(voluntary_reasons)))
ax.set_yticklabels(labels)
ax.set_xlabel('Number of Departures')
ax.set_title('Voluntary Termination Reasons', fontsize=14, fontweight='bold')

# Add value labels
for i, val in enumerate(values):
    pct = (val / len(voluntary)) * 100
    ax.text(val + 1, i, f'{val} ({pct:.0f}%)', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Key Recommendations

Based on the analysis above, here are actionable recommendations:

In [ ]:
print("\n" + "="*60)
print("KEY RECOMMENDATIONS")
print("="*60)

recommendations = []

# Check regrettable attrition
if regrettable_rate > 5:
    recommendations.append(
        f"1. REDUCE REGRETTABLE ATTRITION ({regrettable_rate:.1f}%)\n"
        "   → Implement stay interviews for high performers\n"
        "   → Review compensation competitiveness\n"
        "   → Create retention bonus program for critical roles"
    )

# Check early attrition
early_term_rate = tenure_analysis.loc['0-3 months', 'attrition_rate']
if early_term_rate > 5:
    recommendations.append(
        f"2. ADDRESS EARLY ATTRITION ({early_term_rate}% leave in 90 days)\n"
        "   → Review hiring process (are we setting realistic expectations?)\n"
        "   → Improve onboarding experience\n"
        "   → Conduct 30/60/90 day check-ins with new hires"
    )

# Check high-attrition departments
if len(high_attrition_depts) > 0:
    dept_list = ', '.join(high_attrition_depts.index.tolist())
    recommendations.append(
        f"3. FOCUS ON HIGH-ATTRITION DEPARTMENTS\n"
        f"   → Departments: {dept_list}\n"
        "   → Conduct manager effectiveness reviews\n"
        "   → Review compensation and workload in these areas"
    )

# Top termination reasons
top_reason = voluntary_reasons.index[0].replace('Voluntary - ', '')
recommendations.append(
    f"4. ADDRESS TOP DEPARTURE REASON: {top_reason.upper()}\n"
    "   → Conduct deeper root cause analysis\n"
    "   → Design targeted retention programs"
)

# Print all recommendations
for rec in recommendations:
    print(f"\n{rec}")

print("\n" + "="*60)
print("Next Analysis: See '02_flight_risk_prediction.ipynb' for")
print("predictive modeling to identify at-risk employees")
print("="*60)